# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 11

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. В датафрейме столбец `registered_at` хранит даты регистрации клиентов (например, `'2025-05-12'`), но имеет тип данных `object` (текстовая строка).
Сможет ли аналитик напрямую вычислить разницу в днях между датами (например, `df['registered_at'] - datetime.now()`)? Какая функция Pandas преобразует столбец в специальный тип даты/времени?

**Ответ:**
Нет, со строками типа `object` математические операции дат вызовут ошибку `TypeError`. Для перевода строк в дату/время используется функция `pd.to_datetime(df['registered_at'])`.

---

#### 1.2. Аналитик применил метод `df['balance'].describe()` к столбцу баланса клиентов и получил:
`mean = 1450.0`, `50% (медиана) = 500.0`, `max = 7500.0`.
О чем говорит такой сильный разрыв между средним арифметическим (1450) и медианой (500)? Почему медиана лучше отражает типичный баланс клиента клуба?

**Ответ:**
Разрыв говорит о наличии небольшого числа клиентов с очень крупным балансом (выбросы/VIP-клиенты), которые сильно завышают среднее арифметическое. Медиана (500 руб.) показывает, что ровно у половины клиентов баланс не превышает 500 руб., поэтому она лучше характеризует типичного посетителя.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["duration_diff"] = df["duration_hours"] - df["duration_hours"].mean()
```

**Ответ:**
Вычисляет разницу между продолжительностью конкретной сессии и средним временем сессии по всему киберклубу.

---

#### 1.4. Почему следующий код фильтрации в Pandas упадет с ошибкой или сработает неверно?
```python
# Ошибочный код:
df[df['balance'] > 1000 & df['city'] == 'Москва']
```
Как правильно записать это условие со скобками и почему круглые скобки обязательны?

**Ответ:**
В Python побитовый оператор `&` имеет более высокий приоритет, чем операции сравнения `>` и `==`. Без скобок интерпретатор пытается вычислить `1000 & df['city']`. Правильная запись: `df[(df['balance'] > 1000) & (df['city'] == 'Москва')]`.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `members.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_11")
df_audit = pd.read_csv(data_dir / "members.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Количество уникальных городов: 5")

Количество строк в таблице: 150
Столбцы датафрейма: ['member_id', 'nickname', 'email', 'city', 'registered_at', 'loyalty_tier', 'balance']
Количество уникальных городов: 5

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `bar_orders.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("bar_orders.csv").parent if os.path.exists("bar_orders.csv") else pathlib.Path(f"../variant_11")
df_data = pd.read_csv(data_dir / "bar_orders.csv", sep=";")

summary = df_data.groupby("category").agg({
    "total_amount": "sum",
    "quantity": "mean"
}).reset_index()

summary["quantity"] = summary["quantity"].round(2)
summary = summary.sort_values(by="total_amount", ascending=False).reset_index(drop=True)
display(summary)

,category,total_amount,quantity
0,Мерч,88600,1.49
1,Напитки,32250,1.50
2,Горячее,24740,1.35
3,Снеки,18440,1.43


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `hardware_pcs` столбец `pc_number` (например, 'PC-01') имеет ограничение `UNIQUE`.
От какой проблемы учета это защищает клуб, и может ли столбец с ограничением `UNIQUE` содержать значение `NULL`?

**Ответ:**
Ограничение `UNIQUE` гарантирует, что в клубе не появятся два компьютера с одинаковым инвентарным номером 'PC-01'. В отличие от `PRIMARY KEY`, столбец с ограничением `UNIQUE` в SQL может содержать значение `NULL` (отсутствие номера).

---

#### 3.2. Чем случайное выполнение команды `DROP TABLE members;` опаснее для работы клуба, чем команда `DELETE FROM members;`?
Сможет ли программа регистрации сразу после этого вставить нового клиента?

**Ответ:**
`DELETE` удаляет только строки, сохраняя структуру таблицы, ее типы и ограничения (программа сможет сразу вставлять новых клиентов). `DROP TABLE` физически удаляет саму таблицу из схемы базы данных — приложение упадет с ошибкой «таблица не существует» (`no such table: members`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT discount_promo, COUNT(*) AS cnt, AVG(duration_hours) AS avg_h, SUM(total_paid) AS rev FROM game_sessions GROUP BY discount_promo;
```

**Ответ:**
Запрос группирует сессии по наличию промокода (`discount_promo`) и рассчитывает число сессий, среднюю длительность и суммарную выручку клуба.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_11")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT order_id, member_id, item_name, category, item_price, quantity, total_amount, order_date
FROM bar_orders
WHERE item_price >= 200 AND quantity >= 2
ORDER BY total_amount DESC, order_date DESC
LIMIT 5;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,order_id,member_id,item_name,category,item_price,quantity,total_amount,order_date
0,6286,131,Мерч-футболка CyberClub,Мерч,1500.0,4,6000.0,2026-02-26 11:15:00
1,6241,127,Мерч-футболка CyberClub,Мерч,1500.0,3,4500.0,2026-03-01 12:00:00
2,6327,104,Кепка CyberClub Team,Мерч,800.0,4,3200.0,2026-01-20 20:15:00
3,6101,62,Мерч-футболка CyberClub,Мерч,1500.0,2,3000.0,2026-02-07 18:15:00
4,6111,111,Коврик для мыши CyberClub Speed,Мерч,900.0,3,2700.0,2026-03-04 12:15:00


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT category, COUNT(*) AS orders_count, SUM(quantity) AS total_quantity, ROUND(AVG(total_amount), 2) AS avg_order_sum
FROM bar_orders
GROUP BY category
HAVING SUM(quantity) > 40
ORDER BY total_quantity DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,category,orders_count,total_quantity,avg_order_sum
0,Напитки,126,189,255.95
1,Снеки,100,143,184.40
2,Мерч,59,88,1501.69
3,Горячее,65,88,380.62


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT p.pc_id, p.pc_number, p.zone_name, COUNT(s.session_id) AS promo_sessions, ROUND(SUM(s.total_paid), 2) AS promo_revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE s.discount_promo = 1
GROUP BY p.pc_id, p.pc_number, p.zone_name
ORDER BY promo_revenue DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,pc_id,pc_number,zone_name,promo_sessions,promo_revenue
0,17,PC-17,Bootcamp,6,4010.0
1,25,PC-25,VIP,3,2736.0
2,22,PC-22,Bootcamp,6,2380.0
3,5,PC-05,Standard,6,2161.0
4,21,PC-21,Bootcamp,5,2137.0
5,15,PC-15,Bootcamp,5,2024.0
6,24,PC-24,VIP,3,1980.0
7,3,PC-03,Standard,6,1822.0
8,20,PC-20,Bootcamp,4,1822.0
9,16,PC-16,Bootcamp,4,1688.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT order_id, member_id, item_name, quantity, total_amount
FROM bar_orders
WHERE quantity > (SELECT AVG(quantity) FROM bar_orders)
ORDER BY quantity DESC, total_amount DESC
LIMIT 10;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,order_id,member_id,item_name,quantity,total_amount
0,6286,131,Мерч-футболка CyberClub,4,6000.0
1,6327,104,Кепка CyberClub Team,4,3200.0
2,6087,68,Чипсы картофельные Lays,4,640.0
3,6040,71,Чай зеленый с мятой,4,480.0
4,6241,127,Мерч-футболка CyberClub,3,4500.0
5,6111,111,Коврик для мыши CyberClub Speed,3,2700.0
6,6199,59,Коврик для мыши CyberClub Speed,3,2700.0
7,6220,98,Кепка CyberClub Team,3,2400.0
8,6258,105,Кепка CyberClub Team,3,2400.0
9,6324,106,Кепка CyberClub Team,3,2400.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Отбор активных компьютеров

**Найденные ошибки:**
1. Синтаксическая ошибка в фильтрации: условия в блоке `WHERE` разделены запятой вместо логического оператора `AND`.

In [9]:
query_51_fixed = '''
SELECT 
    pc_number,
    zone_name,
    hourly_rate,
    status
FROM hardware_pcs
WHERE status = 'Active' AND hourly_rate > 200
ORDER BY hourly_rate DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,pc_number,zone_name,hourly_rate,status
0,PC-26,VIP,420.0,Active
1,PC-23,VIP,400.0,Active
2,PC-24,VIP,400.0,Active
3,PC-25,VIP,380.0,Active
4,PC-17,Bootcamp,270.0,Active
5,PC-20,Bootcamp,270.0,Active
6,PC-15,Bootcamp,250.0,Active
7,PC-16,Bootcamp,250.0,Active
8,PC-21,Bootcamp,250.0,Active
9,PC-18,Bootcamp,230.0,Active


#### 5.2. Средняя стоимость аренды компьютеров по зонам

**Найденные ошибки:**
1. Синтаксический порядок: секция `WHERE` написана после `GROUP BY`, фильтрация групп должна выполняться через `HAVING`.
2. Использование псевдонима: в фильтрации указан alias `avg_rate` вместо агрегатного выражения `AVG(hourly_rate)`.

In [10]:
query_52_fixed = '''
SELECT 
    zone_name,
    COUNT(*) AS pcs_count,
    ROUND(AVG(hourly_rate), 2) AS avg_rate
FROM hardware_pcs
GROUP BY zone_name
HAVING AVG(hourly_rate) > 200
ORDER BY avg_rate DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,zone_name,pcs_count,avg_rate
0,VIP,5,404.00
1,PS5,3,306.67
2,Bootcamp,8,247.50


#### 5.3. Сравнение клиентов со средней длительностью сессий клуба

**Найденные ошибки:**
1. Порядок команд: блок `WHERE` написан после `GROUP BY`, фильтрация агрегатов должна выполняться через `HAVING`.
2. Использование псевдонима: в фильтрации указан alias `avg_member_hours` вместо агрегатной функции `AVG(s.duration_hours)`.
3. Многострочный подзапрос: наличие `GROUP BY game_title` в подзапросе возвращает вектор значений, несовместимый со скалярным оператором `>`.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    m.city,
    ROUND(AVG(s.duration_hours), 2) AS avg_member_hours
FROM members m
JOIN game_sessions s ON m.member_id = s.member_id
GROUP BY m.member_id, m.nickname, m.city
HAVING AVG(s.duration_hours) > (
    SELECT AVG(duration_hours) 
    FROM game_sessions
)
ORDER BY avg_member_hours DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,city,avg_member_hours
0,54,Shadow_934,Санкт-Петербург,4.33
1,10,Sentinel_789,Санкт-Петербург,3.67
2,21,Grizzly_295,Санкт-Петербург,3.50
3,63,Grizzly_134,Казань,3.50
4,41,Bullet_471,Санкт-Петербург,3.33
5,102,Wolf_769,Москва,3.17
6,116,Samurai_856,Казань,3.07
7,49,Raven_136,Москва,3.00
8,95,Viper_733,Москва,3.00
9,53,Wolf_302,Москва,2.90
